In [ ]:
# Cell 0 · Install & Import
!pip install awswrangler yfinance matplotlib --quiet

import awswrangler as wr
import yfinance as yf
import pandas as pd
import numpy as np
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')
print('OK')

In [ ]:
# Cell 1 · Config
S3_BUCKET         = 'gold-lstm-forecast'
BRONZE_PATH       = f's3://{S3_BUCKET}/bronze/xauusd_daily/raw'
TICKER            = 'GC=F'
START_DATE        = '2004-06-01'
END_DATE          = datetime.today().strftime('%Y-%m-%d')
CSV_FALLBACK_PATH = BRONZE_PATH

TIMEFRAMES = {
    '1d': {'interval': '1d',  'filename': 'XAU_1d_data.csv'},
    '1w': {'interval': '1wk', 'filename': 'XAU_1w_data.csv'},
    '1m': {'interval': '1mo', 'filename': 'XAU_1Month_data.csv'},
}

print(f'Bucket : {S3_BUCKET}')
print(f'Ticker : {TICKER}')
print(f'Range  : {START_DATE} -> {END_DATE}')

In [ ]:
# Cell 2 · Helper Functions

def fetch_yfinance(ticker, start, end, interval):
    print(f'  Fetching {ticker} | interval={interval} | {start} -> {end}')
    raw = yf.download(tickers=ticker, start=start, end=end,
                      interval=interval, auto_adjust=True, progress=False)
    if raw.empty:
        raise ValueError('yfinance returned empty DataFrame')
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    df = raw[['Open','High','Low','Close','Volume']].copy()
    df.index.name = 'Date'
    df = df.reset_index()
    df['Date'] = pd.to_datetime(df['Date']).dt.date.astype(str)
    print(f'  -> {len(df)} rows fetched')
    return df


def load_csv_fallback(s3_path):
    print(f'  Loading from S3: {s3_path}')
    df = wr.s3.read_csv(path=s3_path)
    if len(df.columns) == 1:
        df = wr.s3.read_csv(path=s3_path, sep=';')
    col_map = {}
    for c in df.columns:
        cl = c.strip().lower()
        if 'date' in cl or 'time' in cl: col_map[c] = 'Date'
        elif cl == 'open':               col_map[c] = 'Open'
        elif cl == 'high':               col_map[c] = 'High'
        elif cl == 'low':                col_map[c] = 'Low'
        elif cl == 'close':              col_map[c] = 'Close'
        elif 'vol' in cl:                col_map[c] = 'Volume'
    df = df.rename(columns=col_map)
    df['Date'] = df['Date'].astype(str).str.replace('.', '-', regex=False).str[:10]
    print(f'  -> {len(df)} rows loaded')
    return df[['Date','Open','High','Low','Close','Volume']]


def upload_to_bronze(df, s3_path, filename):
    full_path = f'{s3_path}/{filename}'
    wr.s3.to_csv(df=df, path=full_path, index=False)
    return full_path


print('Helper functions OK')

In [ ]:
# Cell 3 · Fetch Data
raw_data = {}

for tf_name, tf_config in TIMEFRAMES.items():
    print(f'\n[{tf_name.upper()}]')
    try:
        df = fetch_yfinance(TICKER, START_DATE, END_DATE, tf_config['interval'])
        source = 'yfinance'
    except Exception as e:
        print(f'  yfinance failed: {e}')
        df = load_csv_fallback(f"{CSV_FALLBACK_PATH}/{tf_config['filename']}")
        source = 'csv_fallback'
    raw_data[tf_name] = {'df': df, 'source': source}
    print(f'  Source : {source} | Rows : {len(df)} | {df["Date"].min()} -> {df["Date"].max()}')

print('\nFetch complete')

In [ ]:
# Cell 4 · Validate
def validate_bronze(df, tf_name):
    REQUIRED = {'Date','Open','High','Low','Close','Volume'}
    errors   = []
    missing  = REQUIRED - set(df.columns)
    if missing:                                    errors.append(f'Missing columns: {missing}')
    if len(df) == 0:                               errors.append('Empty DataFrame')
    if df.isnull().all(axis=1).sum() > 0:          errors.append('Fully-null rows found')
    if pd.to_datetime(df['Date'], errors='coerce').isna().sum() > 0:
                                                   errors.append('Unparseable dates')
    print(f'  [{tf_name.upper()}]', end=' ')
    if errors:
        for e in errors: print(f'ERROR: {e}')
        return False
    print(f'PASSED | {len(df):,} rows | {df["Date"].min()} -> {df["Date"].max()}')
    return True

print('=== Bronze Validation ===')
results    = {tf: validate_bronze(d['df'], tf) for tf, d in raw_data.items()}
all_passed = all(results.values())
print(f'\nAll passed: {all_passed}')

In [ ]:
# Cell 5 · Upload to S3 Bronze
tf_to_filename = {
    '1d': 'XAU_1d_data.csv',
    '1w': 'XAU_1w_data.csv',
    '1m': 'XAU_1Month_data.csv',
}

upload_log = []
if all_passed:
    print('=== Uploading to S3 Bronze ===')
    for tf_name, data in raw_data.items():
        filename = tf_to_filename[tf_name]
        s3_uri   = upload_to_bronze(data['df'], BRONZE_PATH, filename)
        upload_log.append({
            'timeframe'  : tf_name, 'filename': filename,
            'rows'       : len(data['df']),
            'date_start' : data['df']['Date'].min(),
            'date_end'   : data['df']['Date'].max(),
            'source'     : data['source'],
            'uploaded_at': datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S UTC'),
        })
        print(f'  OK  {filename} ({len(data["df"]):,} rows)')
    print(f'\nUpload complete — {len(upload_log)} files')
else:
    print('STOP — fix errors first')

In [ ]:
# Cell 6 · Metadata Log
log_df   = pd.DataFrame(upload_log)
log_path = f's3://{S3_BUCKET}/bronze/logs/collection_log.csv'
wr.s3.to_csv(log_df, log_path, index=False)
print(f'Log saved -> {log_path}')
print(log_df[['timeframe','filename','rows','date_start','date_end','source']].to_string(index=False))

In [ ]:
# Cell 7 · Verify Read-back
verify_df = wr.s3.read_csv(path=f'{BRONZE_PATH}/XAU_1d_data.csv')
print(f'Shape      : {verify_df.shape}')
print(f'Columns    : {verify_df.columns.tolist()}')
print(f'Date range : {verify_df["Date"].min()} -> {verify_df["Date"].max()}')
close_num = pd.to_numeric(verify_df['Close'], errors='coerce')
print(f'Close range: ${close_num.min():.2f} -> ${close_num.max():.2f}')
print()
print(verify_df.head(3).to_string(index=False))

In [ ]:
# Cell 8 · EDA + Stats
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_eda          = verify_df.copy()
df_eda['Date']  = pd.to_datetime(df_eda['Date'])
df_eda['Close'] = pd.to_numeric(df_eda['Close'], errors='coerce')
df_eda['Volume']= pd.to_numeric(df_eda['Volume'], errors='coerce')
df_eda          = df_eda.sort_values('Date').reset_index(drop=True)

split_idx  = int(len(df_eda) * 0.8)
split_date = df_eda['Date'].iloc[split_idx]
x_min      = pd.Timestamp('2004-01-01')
x_max      = pd.Timestamp('2027-01-01')

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
fig.suptitle('XAU/USD — Data Collection Overview', fontsize=13, fontweight='bold')

axes[0].plot(df_eda['Date'], df_eda['Close'], color='#E8A020', lw=0.8)
axes[0].axvline(split_date, color='red', lw=1.5, linestyle='--', alpha=0.8)
axes[0].text(split_date, df_eda['Close'].max()*0.92,
             f' Train/Test\n {split_date.date()}', color='red', fontsize=8)
axes[0].set_ylabel('Close Price (USD)')
axes[0].set_title('Daily Close Price 2004-2026')
axes[0].set_xlim(x_min, x_max)
axes[0].xaxis.set_major_locator(mdates.YearLocator(2))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[0].grid(True, alpha=0.3)

axes[1].bar(df_eda['Date'], df_eda['Volume'], color='#4A90D9', alpha=0.6, width=1.5)
axes[1].set_ylabel('Volume')
axes[1].set_title('Trading Volume')
axes[1].set_xlim(x_min, x_max)
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[1].grid(True, alpha=0.3)

returns = df_eda['Close'].pct_change().dropna() * 100
axes[2].hist(returns, bins=120, color='#5CB85C', alpha=0.75, edgecolor='none')
axes[2].axvline(0, color='black', lw=1, linestyle='--', alpha=0.5)
axes[2].set_xlabel('Daily Return (%)')
axes[2].set_ylabel('Frequency')
axes[2].set_title(f'Daily Return Distribution | mean={returns.mean():.3f}% | std={returns.std():.3f}%')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('01_data_collection_overview.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Total rows  : {len(df_eda):,} trading days  ({df_eda["Date"].dt.year.min()}-{df_eda["Date"].dt.year.max()})')
print(f'Missing     : {df_eda[["Open","High","Low","Close","Volume"]].isnull().sum().sum()}')
print(f'Growth      : +{((df_eda["Close"].iloc[-1]/df_eda["Close"].iloc[0])-1)*100:.0f}% over {df_eda["Date"].dt.year.max() - df_eda["Date"].dt.year.min()} years')
print(f'Train       : {split_idx:,} rows  ->  {df_eda["Date"].dt.year.min()} to {split_date.year}')
print(f'Test        : {len(df_eda)-split_idx:,} rows  ->  {split_date.year+1} to {df_eda["Date"].dt.year.max()}')
print(f'Max price   : ${df_eda["Close"].max():,.2f}  ({df_eda.loc[df_eda["Close"].idxmax(), "Date"].year})')